# CS 521 HW3, Problem 3: Multi-Norm Adversarial Training

Use Google Colab with a GPU: **Runtime > Change runtime type > T4 GPU**.

All the code is in this notebook. Complete each place that has a `TODO` and `raise NotImplementedError`.
Do not change the data, the model, the attacks, the evaluation or the default hyperparameters,
unless the homework tells you to change them.

**Threat model.** For each test image $x$, the perturbation region is
$B_\infty(x, 8/255) \cup B_2(x, 0.5) \cup B_1(x, 12)$.
The union accuracy is the percentage of test images that are robust against all three attacks simultaneously.

# Set up for dataset and model

Download the base model, a subset of 10,000 CIFAR-10 training images (1,000 for each class),
and the first 1,000 CIFAR-10 test images from Google Drive.

In [1]:
!gdown 17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes -O linf_base.pt
!gdown 1LoY_o51zNQkrkfvAlGE-44g9G-c38OJe -O cifar10_train_10k.pt
!gdown 1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA -O cifar10_test_1k.pt

Using cookies from /home/dwip2/.cache/gdown/cookies.txt


Downloading...
From: https://drive.google.com/uc?id=17vWkPpVRFUYAxWv3_NEj1gV7Lxh-Cnes
To: /srv/condor/execute/dir_2609564/vlane_cs521hw3-p3seg1-rerun-20261027/work/linf_base.pt

  0%|                                              | 0.00/11.2M [00:00<?, ?B/s]


 33%|████████████                         | 3.67M/11.2M [00:00<00:00, 36.1MB/s]


100%|█████████████████████████████████████| 11.2M/11.2M [00:00<00:00, 61.9MB/s]


Using cookies from /home/dwip2/.cache/gdown/cookies.txt


...


 61%|██████████████████████▋              | 18.9M/30.8M [00:00<00:00, 97.0MB/s]


100%|██████████████████████████████████████| 30.8M/30.8M [00:00<00:00, 106MB/s]
100%|██████████████████████████████████████| 30.8M/30.8M [00:00<00:00, 101MB/s]


Using cookies from /home/dwip2/.cache/gdown/cookies.txt


Downloading...
From: https://drive.google.com/uc?id=1bElxIZ6A708IOGaTMcDq_AsCpC9QQseA
To: /srv/condor/execute/dir_2609564/vlane_cs521hw3-p3seg1-rerun-20261027/work/cifar10_test_1k.pt

  0%|                                              | 0.00/3.08M [00:00<?, ?B/s]


 51%|██████████████████▉                  | 1.57M/3.08M [00:00<00:00, 15.0MB/s]
100%|█████████████████████████████████████| 3.08M/3.08M [00:00<00:00, 25.0MB/s]


In [2]:
import time

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

NORMS = ("linf", "l2", "l1")
EPS = {"linf": 8 / 255, "l2": 0.5, "l1": 12.0}

device: cuda


In [3]:
def load_data(path):
    # The files store images as uint8. Convert them to floats in [0, 1] on the GPU.
    x, y = torch.load(path)
    return x.float().div(255).to(device), y.to(device)


# Training data: 10,000 images. Test data: the first 1,000 CIFAR-10 test images.
x_train, y_train = load_data("cifar10_train_10k.pt")
x_test, y_test = load_data("cifar10_test_1k.pt")
print("train:", tuple(x_train.shape), "test:", tuple(x_test.shape))


def augment(x):
    # Random horizontal flip and random crop (padding 4) for each image.
    n = x.shape[0]
    flip = torch.rand(n, device=x.device) < 0.5
    x = torch.where(flip.view(-1, 1, 1, 1), x.flip(3), x)
    padded = F.pad(x, (4, 4, 4, 4))
    i = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 32, 1)
    j = torch.randint(0, 9, (n, 1, 1), device=x.device) + torch.arange(32, device=x.device).view(1, 1, 32)
    return padded[torch.arange(n, device=x.device).view(n, 1, 1), :, i, j].permute(0, 3, 1, 2).contiguous()


def batches(x, y, batch_size, shuffle):
    idx = torch.randperm(len(x), device=x.device) if shuffle else torch.arange(len(x), device=x.device)
    for k in range(0, len(x), batch_size):
        j = idx[k:k + batch_size]
        yield x[j], y[j]

train: (10000, 3, 32, 32) test: (1000, 3, 32, 32)


In [4]:
# PreActResNet-18 with half width (2.8M parameters). The input is an image in [0, 1].
class PreActBlock(nn.Module):
    def __init__(self, in_planes, planes, stride):
        super().__init__()
        self.bn1 = nn.BatchNorm2d(in_planes)
        self.conv1 = nn.Conv2d(in_planes, planes, 3, stride, 1, bias=False)
        self.bn2 = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, 3, 1, 1, bias=False)
        self.shortcut = None
        if stride != 1 or in_planes != planes:
            self.shortcut = nn.Conv2d(in_planes, planes, 1, stride, bias=False)

    def forward(self, x):
        out = F.relu(self.bn1(x))
        shortcut = self.shortcut(out) if self.shortcut is not None else x
        out = self.conv1(out)
        out = self.conv2(F.relu(self.bn2(out)))
        return out + shortcut


class PreActResNet18(nn.Module):
    def __init__(self, width=32, num_classes=10):
        super().__init__()
        self.register_buffer("mean", torch.tensor([0.4914, 0.4822, 0.4465]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.2471, 0.2435, 0.2616]).view(1, 3, 1, 1))
        self.conv1 = nn.Conv2d(3, width, 3, 1, 1, bias=False)
        layers, in_planes = [], width
        for mult, stride in [(1, 1), (1, 1), (2, 2), (2, 1), (4, 2), (4, 1), (8, 2), (8, 1)]:
            layers.append(PreActBlock(in_planes, mult * width, stride))
            in_planes = mult * width
        self.layers = nn.Sequential(*layers)
        self.bn = nn.BatchNorm2d(in_planes)
        self.linear = nn.Linear(in_planes, num_classes)

    def forward(self, x):
        out = self.layers(self.conv1((x - self.mean) / self.std))
        out = F.adaptive_avg_pool2d(F.relu(self.bn(out)), 1).flatten(1)
        return self.linear(out)


def make_model():
    return PreActResNet18().to(device)


def load_base_model():
    model = make_model()
    model.load_state_dict(torch.load("linf_base.pt", map_location=device))
    return model

# Implement the Attacks

Each attack is untargeted, uses the cross-entropy loss, and returns `x_adv` with
$\|x_{adv} - x\|_p \leq \epsilon$ and $x_{adv} \in [0, 1]$. The attacks must not change the mode (`train` or `eval`) of the model.

* `pgd_linf`: start at a uniformly random point in the $\ell_\infty$ ball. For each step, add `alpha * sign(grad)`,
  clip the perturbation to $[-\epsilon, \epsilon]$, and clip the image to $[0, 1]$.
* `pgd_l2`: start at a random point in the $\ell_2$ ball. For each step, add `alpha * grad / ||grad||_2`,
  scale the perturbation back to the $\ell_2$ ball if its norm is larger than $\epsilon$, and clip the image to $[0, 1]$.

You can use your code from HW1. The $\ell_1$ attack (sparse $\ell_1$ descent, Tramer & Boneh 2019) is complete.

In [ ]:
def clip_box(x, delta):
    # Make x + delta an image in [0, 1]. This never increases |delta_i|.
    return (x + delta).clamp(0, 1) - x


def input_grad(model, x, delta, y):
    delta = delta.detach().requires_grad_(True)
    loss = F.cross_entropy(model(x + delta), y, reduction="sum")
    (grad,) = torch.autograd.grad(loss, delta)
    return grad.detach()


def l2_norm(v):
    return v.flatten(1).norm(dim=1).view(-1, 1, 1, 1)


def pgd_linf(model, x, y, eps, alpha, steps, rand_init=True):
    delta = torch.empty_like(x).uniform_(-eps, eps) if rand_init else torch.zeros_like(x)
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = clip_box(x, (delta + alpha * grad.sign()).clamp(-eps, eps))
    return (x + delta).detach()


def pgd_l2(model, x, y, eps, alpha, steps, rand_init=True):
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.randn_like(x)
        radius = torch.rand(x.shape[0], 1, 1, 1, device=x.device, dtype=x.dtype)
        radius = eps * radius.pow(1.0 / x[0].numel())
        delta = delta / l2_norm(delta).clamp_min(1e-12) * radius
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y)
        delta = delta + alpha * grad / l2_norm(grad).clamp_min(1e-12)
        scale = (eps / l2_norm(delta).clamp_min(1e-12)).clamp(max=1)
        delta = clip_box(x, delta * scale)
    return (x + delta).detach()

In [6]:
def project_l1(delta, eps):
    # Projection of each image of delta onto the L1 ball of radius eps (Duchi et al., 2008).
    v = delta.flatten(1)
    a = v.abs()
    outside = a.sum(dim=1) > eps
    if not outside.any():
        return delta
    mu, _ = a.sort(dim=1, descending=True)
    cumsum = mu.cumsum(dim=1)
    idx = torch.arange(1, v.shape[1] + 1, device=v.device, dtype=v.dtype)
    rho = ((mu - (cumsum - eps) / idx > 0) * idx).argmax(dim=1, keepdim=True)
    theta = (cumsum.gather(1, rho) - eps) / (rho + 1).to(v.dtype)
    w = (a - theta).clamp_min(0) * v.sign()
    return torch.where(outside.unsqueeze(1), w, v).view(delta.shape)


def pgd_l1(model, x, y, eps, alpha, steps, rand_init=True, k=20):
    # Each step changes only the k coordinates with the largest gradient.
    # Coordinates at the image boundary that point outward are skipped.
    delta = torch.zeros_like(x)
    if rand_init:
        delta = torch.distributions.Laplace(0.0, 1.0).sample(x.shape).to(x.device)
        r = torch.rand(x.shape[0], 1, 1, 1, device=x.device)
        delta = delta / delta.flatten(1).abs().sum(1).view(-1, 1, 1, 1) * eps * r
    delta = clip_box(x, delta)
    for _ in range(steps):
        grad = input_grad(model, x, delta, y).flatten(1)
        x_cur = (x + delta).flatten(1)
        blocked = ((grad > 0) & (x_cur >= 1)) | ((grad < 0) & (x_cur <= 0))
        grad = grad.masked_fill(blocked, 0)
        top = grad.abs().topk(k, dim=1).indices
        step = torch.zeros_like(grad).scatter(1, top, grad.gather(1, top).sign())
        delta = clip_box(x, delta + alpha * (step / k).view_as(x))
        delta = project_l1(delta, eps)
    return (x + delta).detach()


ATTACKS = {"linf": pgd_linf, "l2": pgd_l2, "l1": pgd_l1}


def attack(model, x, y, norm, steps=10):
    # The attack for one norm with the default eps and step size.
    alpha = EPS["l1"] / steps ** 0.5 if norm == "l1" else 2.5 * EPS[norm] / steps
    return ATTACKS[norm](model, x, y, EPS[norm], alpha, steps)

# Evaluate Single and Multi-Norm Robust Accuracy

`test_model` gives the clean accuracy, the robust accuracy for each norm, and the union accuracy (in %) on the
1,000 test images. An image is robust for a norm if the model is correct on the clean image and on the result
of each restart of the attack. An image is robust for the union if it is robust for all three norms simultaneously.
Use the default settings (`steps=50, restarts=2`) for all the numbers in your report.

In [7]:
def test_model(model, steps=50, restarts=2, batch_size=250):
    model.eval()
    torch.manual_seed(0)  # the same random starts of the attacks for each model, so the results are comparable
    start = time.time()
    correct = {k: 0 for k in ("clean", *NORMS, "union")}
    for i in range(0, len(x_test), batch_size):
        x, y = x_test[i:i + batch_size], y_test[i:i + batch_size]
        with torch.no_grad():
            clean = model(x).argmax(1) == y
        union = clean.clone()
        for norm in NORMS:
            robust = clean.clone()
            for _ in range(restarts):
                x_adv = attack(model, x, y, norm, steps)
                with torch.no_grad():
                    robust &= model(x_adv).argmax(1) == y
            correct[norm] += robust.sum().item()
            union &= robust
        correct["clean"] += clean.sum().item()
        correct["union"] += union.sum().item()
    acc = {k: 100.0 * v / len(y_test) for k, v in correct.items()}
    print("  ".join(f"{k} {v:.1f}" for k, v in acc.items()) + f"   ({time.time() - start:.0f} s)")
    return acc

In [8]:
# The base model: L-infinity PGD adversarial training for 30 epochs on all 50,000 training images.
test_model(load_base_model())

clean 78.7  linf 48.3  l2 60.1  l1 22.9  union 22.7   (18 s)


# Adversarial Training

`adv_train(model, loss_fn, ...)` fine-tunes `model` for `epochs` epochs on the training subset.
For each batch, it calls `loss_fn(model, x, y, **kwargs)` and does one optimizer step on the loss.

A loss function makes the adversarial examples itself. Put the model in `eval` mode for the attack
(so that the attack does not change the BatchNorm statistics), and in `train` mode before you calculate the loss.

If `gp_beta` is set, `adv_train` does the gradient projection of RAMP (part (b)) before the optimizer step.

In [9]:
def adv_train(model, loss_fn, epochs=10, lr=0.01, batch_size=128, gp_beta=None, seed=0, **kwargs):
    torch.manual_seed(seed)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=5e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs * (len(x_train) // batch_size))
    params = list(model.parameters())
    start = time.time()
    for epoch in range(epochs):
        losses = []
        for x, y in batches(x_train, y_train, batch_size, shuffle=True):
            if len(y) < batch_size:  # skip the last, smaller batch
                continue
            x = augment(x)
            loss = loss_fn(model, x, y, **kwargs)
            opt.zero_grad()
            if gp_beta is None:
                loss.backward()
            else:
                # RAMP (part (b)): combine the adversarial gradient g_a and the natural gradient g_n
                # for each parameter tensor. gradient_projection is defined in part (b).
                g_a = torch.autograd.grad(loss, params)
                g_n = torch.autograd.grad(F.cross_entropy(model(x), y), params)
                for p, ga, gn in zip(params, g_a, g_n):
                    p.grad = gp_beta * gradient_projection(gn, ga) + (1 - gp_beta) * ga
            opt.step()
            sched.step()
            losses.append(loss.item())
        print(f"epoch {epoch + 1}: loss {np.mean(losses):.3f} ({time.time() - start:.0f} s)")
    model.eval()
    return model

# Part (a): Single-Norm Adversarial Training

Complete `single_norm_loss`: attack the batch with the norm `norm` (use `attack`), and return the
cross-entropy loss of the model on the adversarial images.

In [10]:
def single_norm_loss(model, x, y, norm):
    model.eval()
    x_adv = attack(model, x, y, norm)
    model.train()
    return F.cross_entropy(model(x_adv), y)

In [11]:
model_linf = adv_train(load_base_model(), single_norm_loss, norm="linf")
test_model(model_linf)

epoch 1: loss 1.199 (7 s)


epoch 2: loss 1.180 (14 s)


epoch 3: loss 1.154 (21 s)


epoch 4: loss 1.132 (28 s)


epoch 5: loss 1.098 (34 s)


...

epoch 7: loss 1.042 (48 s)


epoch 8: loss 1.027 (55 s)


epoch 9: loss 1.012 (62 s)


epoch 10: loss 1.004 (69 s)


clean 79.0  linf 45.0  l2 57.7  l1 19.8  union 19.6   (18 s)


In [12]:
model_l2 = adv_train(load_base_model(), single_norm_loss, norm="l2")
test_model(model_l2)

epoch 1: loss 0.890 (7 s)


epoch 2: loss 0.823 (14 s)


epoch 3: loss 0.778 (21 s)


epoch 4: loss 0.737 (28 s)


epoch 5: loss 0.704 (34 s)


...

epoch 7: loss 0.622 (48 s)


epoch 8: loss 0.598 (55 s)


epoch 9: loss 0.580 (62 s)


epoch 10: loss 0.573 (69 s)


clean 82.2  linf 32.8  l2 60.5  l1 28.6  union 25.1   (18 s)


In [13]:
model_l1 = adv_train(load_base_model(), single_norm_loss, norm="l1")
test_model(model_l1)

epoch 1: loss 1.232 (8 s)


epoch 2: loss 0.851 (16 s)


epoch 3: loss 0.624 (23 s)


epoch 4: loss 0.450 (31 s)


epoch 5: loss 0.352 (39 s)


...

epoch 7: loss 0.215 (55 s)


epoch 8: loss 0.194 (62 s)


epoch 9: loss 0.171 (70 s)


epoch 10: loss 0.167 (78 s)


clean 83.7  linf 0.0  l2 0.0  l1 25.1  union 0.0   (18 s)


# Part (b): Multi-Norm Adversarial Training (RAMP)

* `ramp_loss`: attack with $q = \ell_\infty$ and $r = \ell_1$. The loss is the mean of
  $\max(\mathcal{L}_{CE}(x^q), \mathcal{L}_{CE}(x^r))$ plus `lam` times $\mathcal{L}_{KL} = KL(p_q \| p_r)$.
  Calculate $\mathcal{L}_{KL}$ only on the images that the model classifies correctly under the $\ell_q$ attack.
* `gradient_projection(g_n, g_a)`: $\cos(g_n, g_a) \cdot g_n$ if $\cos(g_n, g_a) > 0$, else $0$.

`adv_train(..., gp_beta=0.5)` then uses `p.grad = gp_beta * GP(g_n, g_a) + (1 - gp_beta) * g_a` for each parameter
tensor, where $g_a$ is the gradient of `ramp_loss` and $g_n$ is the gradient of the cross-entropy loss on the clean images.

In [ ]:
def ramp_loss(model, x, y, lam=0.5):
    model.eval()
    x_q = attack(model, x, y, "linf")
    x_r = attack(model, x, y, "l1")
    model.train()
    z_q, z_r = model(x_q), model(x_r)
    ce = torch.maximum(F.cross_entropy(z_q, y, reduction="none"),
                       F.cross_entropy(z_r, y, reduction="none")).mean()
    # KL(p_q || p_r), averaged over the images that the model classifies correctly under the l_q attack
    log_q, log_r = F.log_softmax(z_q, 1), F.log_softmax(z_r, 1)
    kl = (log_q.exp() * (log_q - log_r)).sum(1)
    mask = (z_q.argmax(1) == y).float()
    return ce + lam * (kl * mask).sum() / mask.sum().clamp_min(1)


def gradient_projection(g_n, g_a):
    denominator = g_n.norm() * g_a.norm()
    cosine = (g_n * g_a).sum() / denominator.clamp_min(torch.finfo(g_n.dtype).tiny)
    return cosine.clamp(min=0, max=1) * g_n

In [15]:
model_ramp = adv_train(load_base_model(), ramp_loss, gp_beta=0.5)
test_model(model_ramp)

epoch 1: loss 1.430 (15 s)


epoch 2: loss 1.335 (30 s)


epoch 3: loss 1.299 (46 s)


epoch 4: loss 1.268 (61 s)


epoch 5: loss 1.253 (76 s)


...

epoch 7: loss 1.199 (106 s)


epoch 8: loss 1.183 (121 s)


epoch 9: loss 1.173 (137 s)


epoch 10: loss 1.165 (152 s)


clean 78.9  linf 42.2  l2 62.6  l1 45.1  union 39.7   (18 s)


# Part (c): Your Method

Write the training loss of your method in `my_loss`, and train with `adv_train(load_base_model(), my_loss)`.
If your method needs a different update (not the gradient of one loss), write your own copy of `adv_train`.
Run your method and RAMP with `seed=0` and `seed=1`.
Then evaluate your best model again with stronger attacks: `test_model(model, steps=200, restarts=5)`.

In [16]:
def envelope_loss(logits, y):
    target = logits.gather(2, y[None, :, None].expand(logits.shape[0], -1, 1))
    margins = logits - target
    return margins.max(dim=0).values.logsumexp(dim=1).mean()


def my_loss(model, x, y):
    model.eval()
    adversarial = [attack(model, x, y, p) for p in NORMS]
    model.train()
    logits = torch.stack([model(a) for a in adversarial])
    return envelope_loss(logits, y)

In [17]:
model_mine = adv_train(load_base_model(), my_loss)
test_model(model_mine)

epoch 1: loss 1.420 (21 s)


epoch 2: loss 1.314 (42 s)


epoch 3: loss 1.284 (63 s)


epoch 4: loss 1.257 (84 s)


epoch 5: loss 1.226 (105 s)


...

epoch 7: loss 1.167 (147 s)


epoch 8: loss 1.150 (168 s)


epoch 9: loss 1.136 (189 s)


epoch 10: loss 1.130 (210 s)


clean 76.4  linf 43.0  l2 61.9  l1 47.5  union 41.6   (18 s)


Part (c)(iii): the second seed of RAMP and of CAME.

In [18]:
model_ramp_1 = adv_train(load_base_model(), ramp_loss, gp_beta=0.5, seed=1)
test_model(model_ramp_1)
model_mine_1 = adv_train(load_base_model(), my_loss, seed=1)
test_model(model_mine_1)

epoch 1: loss 1.433 (16 s)


epoch 2: loss 1.327 (31 s)


epoch 3: loss 1.297 (46 s)


epoch 4: loss 1.275 (61 s)


epoch 5: loss 1.253 (77 s)


...

epoch 7: loss 1.167 (146 s)


epoch 8: loss 1.152 (167 s)


epoch 9: loss 1.140 (188 s)


epoch 10: loss 1.131 (209 s)


clean 76.7  linf 44.0  l2 62.5  l1 48.8  union 42.5   (18 s)


CAME takes more time per epoch than RAMP. For the same training time,
RAMP also trains for 14 epochs (14 = ceil(10 x measured time ratio)).

In [19]:
model_ramp14 = adv_train(load_base_model(), ramp_loss, epochs=14, gp_beta=0.5)
test_model(model_ramp14)
model_ramp14_1 = adv_train(load_base_model(), ramp_loss, epochs=14, gp_beta=0.5, seed=1)
test_model(model_ramp14_1)

epoch 1: loss 1.430 (16 s)


epoch 2: loss 1.336 (31 s)


epoch 3: loss 1.302 (46 s)


epoch 4: loss 1.273 (61 s)


epoch 5: loss 1.261 (76 s)


...

epoch 11: loss 1.134 (167 s)


epoch 12: loss 1.125 (182 s)


epoch 13: loss 1.121 (197 s)


epoch 14: loss 1.115 (212 s)


clean 79.0  linf 41.6  l2 61.4  l1 44.8  union 39.9   (18 s)


Part (c)(iv), ablation: one worst norm for the whole cross-entropy of each image (MAX-CE).

In [20]:
def ablation_loss(model, x, y):
    model.eval()
    adversarial = [attack(model, x, y, p) for p in NORMS]
    model.train()
    losses = torch.stack([F.cross_entropy(model(a), y, reduction="none") for a in adversarial])
    return losses.max(0).values.mean()

In [21]:
model_ablation = adv_train(load_base_model(), ablation_loss)
test_model(model_ablation)
model_ablation_1 = adv_train(load_base_model(), ablation_loss, seed=1)
test_model(model_ablation_1)

epoch 1: loss 1.404 (21 s)


epoch 2: loss 1.299 (42 s)


epoch 3: loss 1.269 (63 s)


epoch 4: loss 1.243 (84 s)


epoch 5: loss 1.213 (105 s)


...

epoch 7: loss 1.154 (147 s)


epoch 8: loss 1.137 (168 s)


epoch 9: loss 1.126 (189 s)


epoch 10: loss 1.117 (210 s)


clean 76.6  linf 44.4  l2 62.4  l1 48.0  union 42.6   (18 s)


Part (c)(vi): stronger attacks on the better seed of CAME (seed 1).

In [22]:
test_model(model_mine_1, steps=200, restarts=5)

clean 76.7  linf 43.9  l2 62.5  l1 46.3  union 41.6   (175 s)
